# Featurizer Tutorial: Deep Nesting

This notebook demonstrates feature generation across multiple levels of entity relationships using a retail supply chain scenario:
- **Stores** (target, depth 0)
- **Orders** (depth 1)
- **OrderItems** (depth 2) → **Products** (depth 2)
- **Suppliers** (depth 3)

We'll see how features propagate up through the entity graph with `max_depth=3`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

if not Path("data.db").exists():
    exec(open("create_data.py").read())

## 2. The Entity Graph

This example has a deeper entity graph than Examples 01 and 02. The relationships form a chain:

```
Stores ←── Orders ←── OrderItems ──→ Products ←── Suppliers
```

With `max_depth=3`, Featurizer traverses all the way from Stores down to Suppliers.

In [ ]:
with open("config.yaml") as f:
    print(f.read())

## 3. Create Featurizer

Let's load the configuration and see how the entity graph is structured.

In [ ]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target entity: {featurizer.target.alias}")
print(f"Max depth: {featurizer.max_depth}")
print(f"Intervals: {featurizer.intervals}")
print(f"Entities: {len(list(featurizer.entities))}")
print(f"Relationships: {len(featurizer.relationships)}")

In [ ]:
print("Entity Graph:")
for rel in featurizer.relationships:
    print(
        f"  {rel.parent.alias}.{rel.parent_key} ←── {rel.child.alias}.{rel.child_key}"
    )

## 4. Feature Propagation Across Depths

At each depth level, Featurizer aggregates child features and applies transformations. Let's examine how features are distributed across entities.

In [ ]:
print("Features by entity:")
for entity_alias, features in featurizer.features.items():
    print(f"\n  {entity_alias}: {len(features)} features")
    sample = sorted(features, key=lambda f: f.name)[:5]
    for feat in sample:
        print(f"    - {feat.name}")

## 5. The CTE Chain

With deep nesting, the generated SQL has CTEs for each entity at each depth level. The naming pattern is:
- `<entity>_synth` — joins aggregated and direct features
- `<entity>_transform` — applies transformations
- `<child>_aggs_for_<parent>` — aggregation CTE

Let's see the full CTE structure.

In [ ]:
print(f"Number of CTEs: {len(featurizer.ctes)}")
print("\nCTE names:")
for cte in featurizer.ctes:
    lines = cte.strip().split("\n")
    for line in lines:
        if " as (" in line:
            name = line.split(" as (")[0].strip().lstrip("-").strip()
            print(f"  - {name}")
            break

## 6. Generated SQL

The full SQL query shows the depth-3 traversal with lateral joins and time-windowed aggregations.

In [ ]:
sql = featurizer.query
print("Generated SQL Query:")
print("=" * 80)
print(sql)
print("=" * 80)
print(f"\nSQL length: {len(sql):,} characters")

## 7. Feature Count Growth

Deep nesting causes a combinatorial explosion of features. Each depth level multiplies the feature count by the number of aggregations and transformations.

In [ ]:
target_features = featurizer.features[featurizer.target.alias]
print(f"Total target features: {len(target_features)}")

# Analyze feature names
depth_indicators = {
    "Direct (stores)": [f for f in target_features if "orders" not in f.name.lower()],
    "Depth 1 (orders)": [
        f
        for f in target_features
        if "orders" in f.name.lower() and "order_items" not in f.name.lower()
    ],
    "Depth 2+ (items/products)": [
        f
        for f in target_features
        if "order_items" in f.name.lower() or "products" in f.name.lower()
    ],
}

for label, feats in depth_indicators.items():
    print(f"  {label}: ~{len(feats)} features")

## 8. Summary

In this tutorial, we learned:

1. **Deep entity graphs**: How to configure multi-level relationships (Stores → Orders → OrderItems → Products → Suppliers)
2. **Feature propagation**: How aggregated features at each depth level become inputs to the next level
3. **CTE chain**: The naming pattern for CTEs at each entity and depth
4. **Feature explosion**: How feature count grows combinatorially with depth, intervals, and primitives

### Performance Note
Deep nesting (depth > 3) can generate very large SQL queries. Consider:
- Reducing `max_depth` for initial exploration
- Using fewer `intervals`
- Selecting specific aggregations/transformations rather than defaults

In [ ]:
print("Deep Nesting Summary")
print("=" * 40)
print(f"Target: {featurizer.target.alias}")
print(f"Depth: {featurizer.max_depth}")
print(f"Intervals: {', '.join(featurizer.intervals)}")
print(f"Entities: {len(list(featurizer.entities))}")
print(f"Relationships: {len(featurizer.relationships)}")
print(f"Total features: {len(target_features)}")
print(f"SQL length: {len(sql):,} characters")
print(f"CTEs generated: {len(featurizer.ctes)}")